# SQLite with Python

SQLite is a file-based relational database — no server needed. It's built into Python via the `sqlite3` module.

Topics:
- Creating a database and tables
- CRUD: INSERT, SELECT, UPDATE, DELETE
- Querying with WHERE, ORDER BY, GROUP BY, HAVING
- JOIN operations
- Pandas ↔ SQLite integration
- Parameterised queries (SQL injection prevention)

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

DB_PATH = '/tmp/bookstore.db'
Path(DB_PATH).unlink(missing_ok=True)  # fresh start

# Connect (creates file if it doesn't exist)
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row   # rows behave like dicts
cursor = conn.cursor()
print('Connected to', DB_PATH)

## 1. Creating Tables

In [ ]:
# Create authors table
cursor.executescript('''
    CREATE TABLE IF NOT EXISTS authors (
        id      INTEGER PRIMARY KEY AUTOINCREMENT,
        name    TEXT NOT NULL,
        country TEXT,
        born    INTEGER
    );

    CREATE TABLE IF NOT EXISTS books (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        title       TEXT NOT NULL,
        author_id   INTEGER REFERENCES authors(id),
        genre       TEXT,
        pages       INTEGER,
        price       REAL,
        published   TEXT,
        in_stock    INTEGER DEFAULT 1  -- 1=True, 0=False
    );

    CREATE TABLE IF NOT EXISTS sales (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        book_id     INTEGER REFERENCES books(id),
        quantity    INTEGER,
        sale_date   TEXT
    );
''')
conn.commit()
print('Tables created')

## 2. INSERT — Adding Data

In [ ]:
# Insert with parameterised queries (ALWAYS use ? placeholders — never f-strings)
authors_data = [
    ('George Orwell',     'UK',    1903),
    ('J.K. Rowling',      'UK',    1965),
    ('Yuval Noah Harari', 'Israel',1976),
    ('Malcolm Gladwell',  'Canada',1963),
    ('Agatha Christie',   'UK',    1890),
]
cursor.executemany('INSERT INTO authors (name, country, born) VALUES (?, ?, ?)', authors_data)

books_data = [
    ('1984',                        1, 'Dystopian',   328, 12.99, '1949-06-08', 1),
    ('Animal Farm',                 1, 'Satire',      112,  8.99, '1945-08-17', 1),
    ('Harry Potter and the SS',     2, 'Fantasy',     309, 14.99, '1997-06-26', 1),
    ('Harry Potter and the CoS',    2, 'Fantasy',     341, 14.99, '1998-07-02', 1),
    ('Sapiens',                     3, 'Non-Fiction', 443, 16.99, '2011-01-01', 1),
    ('Homo Deus',                   3, 'Non-Fiction', 450, 16.99, '2015-09-04', 1),
    ('Outliers',                    4, 'Non-Fiction', 309, 13.99, '2008-11-18', 1),
    ('The Tipping Point',           4, 'Non-Fiction', 301, 12.99, '2000-03-01', 1),
    ('Murder on the Orient Express',5, 'Mystery',     256, 11.99, '1934-01-01', 1),
    ('And Then There Were None',    5, 'Mystery',     272, 11.99, '1939-11-06', 0),
]
cursor.executemany(
    'INSERT INTO books (title, author_id, genre, pages, price, published, in_stock) VALUES (?,?,?,?,?,?,?)',
    books_data
)

# Sales data
import random
random.seed(42)
sales_data = [(random.randint(1,10), random.randint(1,50), f'2024-{m:02d}-{d:02d}')
              for m in range(1,7) for d in [5,12,20,28]]
cursor.executemany('INSERT INTO sales (book_id, quantity, sale_date) VALUES (?,?,?)', sales_data)

conn.commit()
print('Data inserted')

## 3. SELECT — Reading Data

In [ ]:
# Basic SELECT
print('=== All Authors ===')
for row in cursor.execute('SELECT * FROM authors'):
    print(dict(row))

print('\n=== Books under $14 ===')
for row in cursor.execute('SELECT title, price FROM books WHERE price < 14.00 ORDER BY price'):
    print(f"  {row['title']:40s}  ${row['price']:.2f}")

In [ ]:
# Aggregation: GROUP BY + HAVING
print('=== Average price by genre (>1 book) ===')
result = cursor.execute('''
    SELECT genre,
           COUNT(*)       AS book_count,
           AVG(price)     AS avg_price,
           SUM(pages)     AS total_pages
    FROM   books
    GROUP  BY genre
    HAVING COUNT(*) > 1
    ORDER  BY avg_price DESC
''').fetchall()
for row in result:
    print(f"  {row['genre']:15s}  {row['book_count']} books  avg=${row['avg_price']:.2f}")

## 4. JOIN — Combining Tables

In [ ]:
# INNER JOIN: books with their author names
print('=== Books with Author Names ===')
result = cursor.execute('''
    SELECT b.title, a.name AS author, b.genre, b.price
    FROM   books b
    JOIN   authors a ON b.author_id = a.id
    ORDER  BY a.name, b.published
''').fetchall()
for row in result:
    print(f"  {row['author']:25s}  {row['title']:35s}  ${row['price']:.2f}")

In [ ]:
# Multi-table join: total sales per author
print('=== Total Units Sold per Author ===')
result = cursor.execute('''
    SELECT a.name AS author,
           COUNT(DISTINCT b.id)  AS books_sold,
           SUM(s.quantity)        AS total_units,
           ROUND(SUM(s.quantity * b.price), 2) AS total_revenue
    FROM   sales s
    JOIN   books   b ON s.book_id = b.id
    JOIN   authors a ON b.author_id = a.id
    GROUP  BY a.name
    ORDER  BY total_revenue DESC
''').fetchall()
for row in result:
    print(f"  {row['author']:25s}  {row['total_units']:4d} units  ${row['total_revenue']:8,.2f}")

## 5. UPDATE and DELETE

In [ ]:
# UPDATE: mark out-of-stock books with price reduction
print('Before:', cursor.execute('SELECT title, price, in_stock FROM books WHERE in_stock=0').fetchall())

cursor.execute('''
    UPDATE books
    SET    price = price * 0.90,
           in_stock = 1
    WHERE  in_stock = 0
''')
conn.commit()
print('After:', cursor.execute('SELECT title, price, in_stock FROM books WHERE title=?',
                               ("And Then There Were None",)).fetchone()['price'])

# DELETE: remove old sales records
cursor.execute("DELETE FROM sales WHERE sale_date < '2024-03-01'")
conn.commit()
print(f'Remaining sales: {cursor.execute("SELECT COUNT(*) FROM sales").fetchone()[0]}')

## 6. Pandas ↔ SQLite Integration

In [ ]:
# Read query result directly into a DataFrame
df = pd.read_sql('''
    SELECT b.title, a.name AS author, b.genre, b.price,
           COALESCE(SUM(s.quantity), 0) AS total_sold
    FROM   books b
    JOIN   authors a ON b.author_id = a.id
    LEFT JOIN sales s ON b.id = s.book_id
    GROUP  BY b.id
    ORDER  BY total_sold DESC
''', conn)

print(df.to_string(index=False))
print('\nAverage price:', df['price'].mean().round(2))
print('Best seller:', df.iloc[0]['title'])

In [ ]:
# Write a DataFrame to SQLite
new_books = pd.DataFrame({
    'title':     ['Deep Work', 'Atomic Habits'],
    'author_id': [4, 4],
    'genre':     ['Non-Fiction', 'Non-Fiction'],
    'pages':     [304, 320],
    'price':     [14.99, 13.99],
    'published': ['2016-01-05', '2018-10-16'],
    'in_stock':  [1, 1]
})
new_books.to_sql('books', conn, if_exists='append', index=False)
print('Inserted', len(new_books), 'new books')
print('Total books:', cursor.execute('SELECT COUNT(*) FROM books').fetchone()[0])

## 7. SQL Injection — Why Use Parameters?

In [ ]:
# DANGEROUS: string formatting in SQL
user_input = "'; DROP TABLE books; --"   # SQL injection attempt
# cursor.execute(f"SELECT * FROM books WHERE title = '{user_input}'")  # NEVER DO THIS

# SAFE: parameterised query — sqlite3 escapes user input
cursor.execute('SELECT * FROM books WHERE title = ?', (user_input,))
result = cursor.fetchall()
print(f'Safe query returned {len(result)} rows (books table still intact)')
print('Book count:', cursor.execute('SELECT COUNT(*) FROM books').fetchone()[0])

In [ ]:
# Always close the connection when done
conn.close()
print('Connection closed')

## Quick Summary

| Operation | SQL Snippet |
|-----------|-------------|
| Create table | `CREATE TABLE t (id INTEGER PRIMARY KEY, ...)` |
| Insert | `INSERT INTO t (col1) VALUES (?)` with params |
| Select | `SELECT col FROM t WHERE cond ORDER BY col LIMIT n` |
| Aggregate | `SELECT col, COUNT(*), AVG(x) FROM t GROUP BY col HAVING ...` |
| Join | `SELECT ... FROM t1 JOIN t2 ON t1.id = t2.fk` |
| Update | `UPDATE t SET col = val WHERE cond` |
| Delete | `DELETE FROM t WHERE cond` |
| Pandas read | `pd.read_sql('SELECT ...', conn)` |
| Pandas write | `df.to_sql('table', conn, if_exists='append')` |

> Always use `?` placeholders for user-supplied values — never f-strings.

**Next →** [08 – EDA Projects](../08-eda-projects/)